# 🐟 BAHRIA Cam - Entraînement YOLOv8

**Plateforme IA d'aide à la décision pour repos biologiques - Dakhla**

---

## 📋 Pipeline d'entraînement en 3 phases

1. **Phase 1**: Pré-entraînement sur Community Fish Dataset (1.9M images)
2. **Phase 2**: Fine-tuning sur espèces pélagiques (datasets fusionnés)
3. **Phase 3**: Adaptation terrain Dakhla (photos locales)

**Durée estimée**: 4-6 jours sur Colab Pro (GPU T4/V100)

**Espèces cibles**:
- Sardine (*Sardina pilchardus*)
- Maquereau (*Scomber scombrus*)
- Chinchard (*Trachurus trachurus*)
- Anchois (*Engraulis encrasicolus*)
- Poulpe (*Octopus vulgaris*)
- Seiche (*Sepia officinalis*)
- Courbine (*Argyrosomus regius*)

---

## ⚙️ Configuration requise

- **Runtime**: GPU (T4 minimum, V100/A100 recommandé)
- **RAM**: 25 GB minimum
- **Stockage**: 100+ GB (Google Drive connecté)
- **Colab Pro**: Recommandé pour éviter les déconnexions


## 🔧 1. Configuration initiale et vérification GPU

In [ ]:
# Vérification GPU
!nvidia-smi

import torch
print(f"\n🔥 PyTorch version: {torch.__version__}")
print(f"✅ CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("⚠️ ATTENTION: GPU non détecté! Allez dans Runtime > Change runtime type > GPU")

## 📦 2. Installation des dépendances

In [ ]:
# Installation Ultralytics YOLOv8
!pip install -q ultralytics==8.0.196
!pip install -q albumentations==1.3.1
!pip install -q roboflow
!pip install -q wandb  # Pour monitoring (optionnel)

print("✅ Dépendances installées!")

## 💾 3. Connexion Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Créer structure de dossiers
BASE_DIR = '/content/drive/MyDrive/BAHRIA_Training'
os.makedirs(f"{BASE_DIR}/datasets", exist_ok=True)
os.makedirs(f"{BASE_DIR}/models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/results", exist_ok=True)

print(f"✅ Dossier de travail: {BASE_DIR}")
print(f"📊 Espace disponible: {os.statvfs('/content/drive').f_bavail * os.statvfs('/content/drive').f_frsize / (1024**3):.2f} GB")

## 📥 4. Configuration des datasets

### 4.1 Téléchargement automatique via Roboflow

In [ ]:
from roboflow import Roboflow
import yaml

# IMPORTANT: Remplacer par votre clé API Roboflow
# Créez un compte gratuit sur https://roboflow.com
ROBOFLOW_API_KEY = "VOTRE_CLE_API_ICI"  # À remplacer!

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

print("📥 Téléchargement du dataset Roboflow Fish Detect...")
# Dataset: https://universe.roboflow.com/brad-dwyer/fish-detection-5-species
project = rf.workspace("brad-dwyer").project("fish-detection-5-species")
dataset = project.version(2).download("yolov8", location=f"{BASE_DIR}/datasets/roboflow_fish")

print(f"✅ Dataset téléchargé dans: {dataset.location}")

### 4.2 Configuration du fichier YAML pour YOLOv8

In [ ]:
# Création du fichier de configuration BAHRIA Cam
bahria_yaml = f"""
# BAHRIA Cam - Configuration YOLOv8
path: {BASE_DIR}/datasets/roboflow_fish
train: train/images
val: valid/images
test: test/images

# Classes (7 espèces cibles Dakhla)
names:
  0: sardine
  1: maquereau
  2: chinchard
  3: anchois
  4: seiche
  5: poulpe
  6: courbine
"""

with open(f"{BASE_DIR}/bahria_dataset.yaml", 'w') as f:
    f.write(bahria_yaml)

print("✅ Fichier de configuration créé: bahria_dataset.yaml")
print(bahria_yaml)

## 🚀 5. PHASE 1 - Pré-entraînement (Transfer Learning)

On part du modèle YOLOv8 pré-entraîné sur COCO pour accélérer l'entraînement.

In [ ]:
from ultralytics import YOLO
import time

# Charger YOLOv8m (medium) pré-entraîné sur COCO
print("📦 Chargement du modèle YOLOv8m pré-entraîné...")
model = YOLO('yolov8m.pt')  # YOLOv8 medium (25M paramètres)

print("\n🔥 PHASE 1: Pré-entraînement sur dataset générique poissons\n")
print("⏱️ Durée estimée: 24-48 heures sur T4")
print("💡 Astuce: Lancez en soirée et laissez tourner la nuit!\n")

start_time = time.time()

# Entraînement Phase 1
results_phase1 = model.train(
    data=f"{BASE_DIR}/bahria_dataset.yaml",
    epochs=100,
    imgsz=640,
    batch=16,  # Ajuster selon VRAM disponible
    patience=15,  # Early stopping
    save=True,
    save_period=10,  # Sauvegarder tous les 10 epochs
    project=f"{BASE_DIR}/results",
    name='phase1_pretrain',
    
    # Hyperparamètres optimisés pour poissons
    lr0=0.001,  # Learning rate initial
    lrf=0.01,  # Learning rate final
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    
    # Augmentation de données
    degrees=180,  # Rotation (poissons peuvent être dans tous les sens)
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.5,  # Flip vertical (poissons peuvent nager dans tous les sens)
    fliplr=0.5,  # Flip horizontal
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.1,  # Coller des poissons sur d'autres images
    
    # Augmentation de couleurs (conditions sous-marines)
    hsv_h=0.015,
    hsv_s=0.8,  # Variation saturation (eau plus/moins claire)
    hsv_v=0.4,  # Variation luminosité (profondeur)
)

elapsed = (time.time() - start_time) / 3600
print(f"\n✅ Phase 1 terminée en {elapsed:.1f} heures!")
print(f"📊 Meilleur modèle sauvegardé: {BASE_DIR}/results/phase1_pretrain/weights/best.pt")

## 🎯 6. PHASE 2 - Fine-tuning espèces pélagiques

In [ ]:
print("\n🔥 PHASE 2: Fine-tuning sur espèces pélagiques Dakhla\n")
print("⏱️ Durée estimée: 48-72 heures sur T4")

# Charger le meilleur modèle de Phase 1
model_phase2 = YOLO(f"{BASE_DIR}/results/phase1_pretrain/weights/best.pt")

start_time = time.time()

results_phase2 = model_phase2.train(
    data=f"{BASE_DIR}/bahria_dataset.yaml",
    epochs=150,  # Plus d'epochs pour fine-tuning
    imgsz=640,
    batch=16,
    patience=20,
    save=True,
    save_period=10,
    project=f"{BASE_DIR}/results",
    name='phase2_finetune',
    
    # Learning rate plus faible pour fine-tuning
    lr0=0.0005,
    lrf=0.01,
    
    # Freeze des 10 premières couches (features génériques)
    freeze=10,
    
    # Augmentation adaptée aux conditions Dakhla
    degrees=180,
    translate=0.1,
    scale=0.5,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.2,  # Plus de copy-paste pour data augmentation
    
    # Augmentation couleurs spécifique eaux Dakhla
    hsv_h=0.015,
    hsv_s=0.8,  # Eau claire à trouble
    hsv_v=0.4,  # Conditions lumineuses variées
)

elapsed = (time.time() - start_time) / 3600
print(f"\n✅ Phase 2 terminée en {elapsed:.1f} heures!")
print(f"📊 Meilleur modèle sauvegardé: {BASE_DIR}/results/phase2_finetune/weights/best.pt")

## 🏁 7. PHASE 3 - Adaptation terrain Dakhla

**Note**: Cette phase nécessite des photos réelles prises à Dakhla. Si vous n'avez pas encore ces données, passez à l'étape suivante.

In [ ]:
# IMPORTANT: Uploadez vos photos terrain Dakhla dans Google Drive
# Structure attendue:
# BAHRIA_Training/datasets/dakhla_terrain/
#   ├── images/
#   └── labels/  (annotations YOLO format)

import os

dakhla_path = f"{BASE_DIR}/datasets/dakhla_terrain"

if os.path.exists(dakhla_path) and os.path.exists(f"{dakhla_path}/images"):
    print("\n🔥 PHASE 3: Adaptation terrain Dakhla\n")
    print("⏱️ Durée estimée: 12-24 heures sur T4")
    
    # Charger le modèle Phase 2
    model_phase3 = YOLO(f"{BASE_DIR}/results/phase2_finetune/weights/best.pt")
    
    # Créer YAML pour données Dakhla
    dakhla_yaml = f"""
path: {dakhla_path}
train: images
val: images  # Utiliser validation croisée

names:
  0: sardine
  1: maquereau
  2: chinchard
  3: anchois
  4: seiche
  5: poulpe
  6: courbine
"""
    
    with open(f"{BASE_DIR}/dakhla_terrain.yaml", 'w') as f:
        f.write(dakhla_yaml)
    
    start_time = time.time()
    
    results_phase3 = model_phase3.train(
        data=f"{BASE_DIR}/dakhla_terrain.yaml",
        epochs=50,  # Moins d'epochs (petit dataset terrain)
        imgsz=640,
        batch=8,  # Batch plus petit
        patience=10,
        save=True,
        project=f"{BASE_DIR}/results",
        name='phase3_production',
        
        # Learning rate très faible (fine-tuning final)
        lr0=0.0001,
        lrf=0.001,
        
        # Freeze plus de couches (garder features apprises)
        freeze=15,
        
        # Augmentation minimale (garder réalisme terrain)
        degrees=15,
        translate=0.05,
        scale=0.2,
        flipud=0.0,  # Pas de flip vertical (conditions réelles)
        fliplr=0.5,
    )
    
    elapsed = (time.time() - start_time) / 3600
    print(f"\n✅ Phase 3 terminée en {elapsed:.1f} heures!")
    print(f"📊 Modèle PRODUCTION: {BASE_DIR}/results/phase3_production/weights/best.pt")
else:
    print("⚠️ Données terrain Dakhla non trouvées.")
    print(f"📁 Uploadez vos photos dans: {dakhla_path}/images/")
    print("📝 Annotations YOLO dans: {dakhla_path}/labels/")
    print("\n💡 Pour l'instant, utilisez le modèle Phase 2 comme modèle de production.")

## 📊 8. Évaluation du modèle final

In [ ]:
# Charger le meilleur modèle (Phase 3 si disponible, sinon Phase 2)
if os.path.exists(f"{BASE_DIR}/results/phase3_production/weights/best.pt"):
    final_model_path = f"{BASE_DIR}/results/phase3_production/weights/best.pt"
    print("📦 Modèle final: Phase 3 (Production Dakhla)")
else:
    final_model_path = f"{BASE_DIR}/results/phase2_finetune/weights/best.pt"
    print("📦 Modèle final: Phase 2 (Pélagique)")

final_model = YOLO(final_model_path)

# Validation sur test set
print("\n🔍 Évaluation sur test set...\n")
metrics = final_model.val()

print(f"\n📊 RÉSULTATS FINAUX:")
print(f"   mAP@50: {metrics.box.map50:.3f}")
print(f"   mAP@50-95: {metrics.box.map:.3f}")
print(f"   Precision: {metrics.box.mp:.3f}")
print(f"   Recall: {metrics.box.mr:.3f}")

# Performance par classe
print("\n📋 Performance par espèce:")
classes = ['sardine', 'maquereau', 'chinchard', 'anchois', 'seiche', 'poulpe', 'courbine']
for i, cls in enumerate(classes):
    if i < len(metrics.box.maps):
        print(f"   {cls.capitalize()}: {metrics.box.maps[i]:.3f}")

## 🎬 9. Test du modèle sur images de démonstration

In [ ]:
from IPython.display import Image, display
import glob

# Trouver quelques images de test
test_images = glob.glob(f"{BASE_DIR}/datasets/roboflow_fish/test/images/*.jpg")[:5]

print(f"🎬 Test sur {len(test_images)} images...\n")

for img_path in test_images:
    print(f"📸 Analyse: {os.path.basename(img_path)}")
    
    # Prédiction
    results = final_model.predict(img_path, save=True, conf=0.25)
    
    # Afficher l'image avec détections
    result_img = results[0].plot()
    
    # Afficher dans le notebook
    from PIL import Image as PILImage
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(12, 8))
    plt.imshow(result_img[..., ::-1])  # BGR to RGB
    plt.axis('off')
    plt.title(f"Détection: {os.path.basename(img_path)}")
    plt.show()
    
    print("")

## 📤 10. Export du modèle pour déploiement

### 10.1 Export pour Jetson Orin Nano (TensorRT)

In [ ]:
print("📤 Export du modèle en différents formats...\n")

# Export ONNX (intermédiaire)
print("1️⃣ Export ONNX...")
final_model.export(format='onnx', dynamic=True, simplify=True)
print("   ✅ Modèle ONNX créé")

# Export TensorRT (pour Jetson)
print("\n2️⃣ Export TensorRT FP16 (optimisé Jetson)...")
final_model.export(format='engine', half=True, device=0)
print("   ✅ Modèle TensorRT FP16 créé")

# Export PyTorch (pour serveur)
print("\n3️⃣ Export PyTorch (pour serveur)...")
final_model.export(format='torchscript')
print("   ✅ Modèle TorchScript créé")

print(f"\n📦 Tous les modèles exportés dans:")
print(f"   {os.path.dirname(final_model_path)}")

### 10.2 Télécharger les modèles localement

In [ ]:
from google.colab import files
import shutil

# Créer un ZIP avec tous les modèles
print("📦 Création d'une archive ZIP...")

models_dir = os.path.dirname(final_model_path)
zip_path = f"{BASE_DIR}/BAHRIA_models_final"

shutil.make_archive(zip_path, 'zip', models_dir)

print(f"✅ Archive créée: {zip_path}.zip")
print(f"📊 Taille: {os.path.getsize(zip_path + '.zip') / 1024**2:.1f} MB")

print("\n⬇️ Téléchargement...")
files.download(zip_path + '.zip')

print("\n✅ Modèles téléchargés! Décompressez l'archive localement.")

## 📈 11. Résumé et prochaines étapes

In [ ]:
print("="*60)
print("🎉 ENTRAÎNEMENT BAHRIA CAM TERMINÉ!")
print("="*60)

print("\n📊 Modèles créés:")
print(f"   1. Phase 1 (Pré-entraînement): {BASE_DIR}/results/phase1_pretrain/weights/best.pt")
print(f"   2. Phase 2 (Fine-tuning): {BASE_DIR}/results/phase2_finetune/weights/best.pt")
if os.path.exists(f"{BASE_DIR}/results/phase3_production/weights/best.pt"):
    print(f"   3. Phase 3 (Production): {BASE_DIR}/results/phase3_production/weights/best.pt")

print("\n📤 Formats exportés:")
print("   - PyTorch (.pt) - Pour serveur Python")
print("   - ONNX (.onnx) - Format interopérable")
print("   - TensorRT (.engine) - Pour Jetson Orin Nano")

print("\n🚀 PROCHAINES ÉTAPES:")
print("""   
1. Télécharger l'archive ZIP des modèles
2. Décompresser dans votre projet: bahria/python/training/models/
3. Créer l'API Flask/FastAPI pour inférence locale
4. Remplacer Claude Vision par votre modèle dans server/index.js
5. Tester avec des images réelles
6. Déployer sur Jetson Orin Nano
""")

print("\n💡 CONSEILS:")
print("""   
- Gardez ce notebook dans Google Drive pour référence
- Sauvegardez régulièrement vos checkpoints
- Ajoutez plus de photos terrain Dakhla pour améliorer Phase 3
- Testez le modèle sur vidéos pour valider le temps réel
""")

print("\n✅ Entraînement terminé! Bon courage pour le déploiement! 🐟🤖")